In [ ]:
from itertools import product

P_B = {1: 0.001, 0: 0.999}
P_E = {1: 0.002, 0: 0.998}

In [ ]:
P_A_given_BE = {
    (1,1): 0.95,
    (1,0): 0.94,
    (0,1): 0.29,
    (0,0): 0.001
}

P_J_given_A= {1: 0.90, 0: 0.05}
P_M_given_A= {1: 0.70, 0: 0.01}

In [ ]:
def joint_prob(B, E, A,J,M):
    p = P_B[B] * P_E[E]
    p *= P_A_given_BE[(B,E)] if A == 1 else (1- P_A_given_BE[(B,E)])
    p *= P_J_given_A[A] if J == 1 else (1- P_J_given_A[A])
    p *= P_M_given_A[A] if M == 1 else (1- P_M_given_A[A])
    return p

In [ ]:
joint_probs = {}
for B,E,A,J,M in product([0,1], repeat = 5):
    joint_probs[(B,E,A,J,M)] = joint_prob(B,E,A,J,M)

In [ ]:
numerator = sum(p for (B,E,A,J,M), p in joint_probs.items() if B == 1 and A == 1)
denominator = sum(p for (B,E,A,J,M), p in joint_probs.items() if A == 1)
P_B1_given_A1 = numerator / denominator

In [ ]:
numerator_E = sum(p for (B,E,A,J,M), p in joint_probs.items() if E == 1 and A == 1)
P_E1_given_A1 = numerator_E / denominator

In [ ]:
numerator_JM = sum(p for (B,E,A,J,M), p in joint_probs.items() if B == 1 and J == 1 and M == 1)
denominator_JM = sum(p for (B,E,A,J,M), p in joint_probs.items() if J == 1 and M == 1)
P_B1_given_J1M1 = numerator_JM / denominator_JM

P_B1_given_A1, P_E1_given_A1, P_B1_given_J1M1

In [ ]:
P_yes = 9 / 14
P_no = 5 / 14

In [ ]:
cpt = {
    'Outlook': {
        'Yes': {'Sunny': 2/9, 'Overcast': 4/9, 'Rain': 3/9},
        'No': {'Sunny': 3/5, 'Overcast': 0/5, 'Rain': 2/5}
    },
    'Temperature': {
        'Yes': {'Hot': 2/9, 'Mild': 4/9, 'Cool': 3/9},
        'No': {'Hot': 2/5, 'Mild': 2/5, 'Cool': 1/5}
    },
    'Humidity': {
        'Yes': {'High': 3/9, 'Normal': 6/9},
        'No': {'High': 4/5, 'Normal': 1/5}
    },
    'Wind': {
        'Yes': {'Strong': 3/9, 'Weak': 6/9},
        'No': {'Strong': 3/5, 'Weak': 2/5}
    }
}

In [ ]:
x = {
    'Outlook': 'Sunny',
    'Temperature' : 'Cool',
    'Humidity' : 'High',
    'Wind': 'Strong'
}


In [ ]:
P_x_given_yes = (
    cpt['Outlook']['Yes'][x['Outlook']] *
    cpt['Temperature']['Yes'][x['Temperature']] *
    cpt['Humidity']['Yes'][x['Humidity']] *
    cpt['Wind']['Yes'][x['Wind']] *
    P_yes
)

In [ ]:
P_x_given_no = (
    cpt['Outlook']['No'][x['Outlook']] *
    cpt['Temperature']['No'][x['Temperature']] *
    cpt['Humidity']['No'][x['Humidity']] *
    cpt['Wind']['No'][x['Wind']] *
    P_no
)

In [ ]:
P_yes_given_x = P_x_given_yes / (P_x_given_yes + P_x_given_no)
P_no_given_x = P_x_given_no / (P_x_given_yes + P_x_given_no)

In [ ]:
P_yes_given_x, P_no_given_x

In [ ]:
from collections import defaultdict
import math

emails = [
    "send us your password",    # spam
    "send us your review",      # ham
    "review your password",     # ham
    "review us",                # spam
    "send your password",       # spam
    "send us your account"      # spam
]
labels = ["spam", "ham", "ham", "spam","spam","spam"]

In [ ]:
vocab = set()
word_counts = {'spam': defaultdict(int), 'ham': defaultdict(int)}
class_counts = {'spam': 0, 'ham': 0}
total_docs = len(emails)

In [ ]:
for text, label in zip(emails, labels):
    words = text.split()
    class_counts[label] += 1
    for word in words:
        vocab.add(word)
        word_counts[label][word] += 1

In [ ]:
alpha = 1
V = len(vocab)
total_words = {
    'spam': sum(word_counts['spam'].values()),
    'ham': sum(word_counts['ham'].values())
}

In [ ]:
new_email = "review us account"
new_words = new_email.split()

In [ ]:
def compute_log_prob(class_label):
    log_prob = math.log(class_counts[class_label] / total_docs)
    for word in new_words:
        count = word_counts[class_label].get(word, 0)
        prob = (count + alpha) / (total_words[class_label] + alpha * V)
        log_prob += math.log(prob)
    return log_prob

log_prob_spam = compute_log_prob('spam')
log_prob_ham = compute_log_prob('ham')

In [ ]:
prob_spam = math.exp(log_prob_spam)
prob_ham = math.exp(log_prob_ham)
total = prob_spam + prob_ham
normalized_probs = {
    'spam': prob_spam / total,
    'ham': prob_ham / total
}

In [ ]:
normalized_probs

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

In [ ]:
from collections import defaultdict
import math

emails = [
    "send us your password",    # spam
    "send us your review",      # ham
    "review your password",     # ham
    "review us",                # spam
    "send your password",       # spam
    "send us your account"      # spam
]
labels = ["spam", "ham", "ham", "spam","spam","spam"]

In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(emails)
y = np.array(labels)

In [ ]:
model = MultinomialNB(alpha=1.0)
model.fit(X, y)

In [ ]:
new_email = ["review us account"]
X_new = vectorizer.transform(new_email)

In [ ]:
probs = model.predict_proba(X_new)[0]
classes = model.classes_
result = dict(zip(classes, probs))
result

In [ ]:
import pickle
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

with open("data/mnist.pkl", "rb") as f:
    train_set, valid_set, test_set = pickle.load(f, encoding="latin1")

X_train, y_train = train_set
X_test, y_test = test_set

In [ ]:
X_train_bin = (X_train > 0.5).astype(int)
X_test_bin = (X_test > 0.5).astype(int)

In [ ]:
clf = BernoulliNB(alpha=0.1)
clf.fit(X_train_bin, y_train)

In [ ]:
y_pred = clf.predict(X_test_bin)
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

accuracy, conf_matrix


In [ ]:
import pandas as pd
import nltk
import string
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
df = pd.read_csv("data/emails.csv")

In [ ]:
texts = df['text']
labels = df['spam']

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

text_clean = texts.apply(preprocess)
                  

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(text_clean, labels, test_size=0.3, random_state=42)

In [ ]:
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

In [ ]:
y_pred = clf.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
sample_emails = ["Free money offer", "Can we meet now?"]
sample_clean = [preprocess(t) for t in sample_emails]
sample_vec = vectorizer.transform(sample_clean)
sample_preds = clf.predict(sample_vec)

accuracy, list(sample_preds)